# Step 1 - Train the model (do this *before* the demo)

Run this once on the GPU box. It does three things:

1. downloads a few hundred real galaxy images from the SDSS public archive
2. trains a small neural network to reverse the blur-and-shrink process
3. saves the trained weights to `weights/galaxy_sr.pt`

Nothing here runs during the live demo. Budget ~10-20 minutes total.

**If the machine has no internet**, this still works - it falls back to
simulated galaxies built from the same equations astronomers use to model
real ones. The demo is just less pretty.

In [ ]:
import torch
from galaxy_sr import data, viz
from galaxy_sr.train import train

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))

## Get the images

`build_dataset` caches everything to `data/` as PNGs, so re-running is cheap
and the demo notebook never needs the network.

In [ ]:
info = data.build_dataset("data", n_train=300, size=256)
info

In [ ]:
# quick look at what we got
import matplotlib.pyplot as plt
imgs = data.load_images("data/train")[:8]
fig, axes = plt.subplots(1, 8, figsize=(20, 3))
for ax, im in zip(axes, imgs):
    ax.imshow(im); ax.axis("off")
plt.show()

## Train

Roughly 3-8 minutes on a modern GPU at these settings. Raise `steps` if you
have time - the gap over the non-AI baseline keeps widening for a while.

In [ ]:
info = train(
    data_dir="data/train",
    out="weights/galaxy_sr.pt",
    steps=6000,       # lower to 2000 for a fast first pass
    batch=16,
    patch=128,
    channels=64,
    blocks=12,
    factor=4,         # 4x upscaling
    sigma=1.6,        # how badly the "small telescope" blurs
    noise=0.012,      # how noisy its detector is
)

## Sanity check before the demo

Look at these. If the AI panel is not visibly better than the ordinary
enlargement, train for more steps before demo day.

In [ ]:
from galaxy_sr.model import load_trained

model, info = load_trained("weights/galaxy_sr.pt")
demo_imgs = data.load_images("data/demo")
res = viz.run_all(model, demo_imgs[0])

viz.compare(res, save="figures/check_compare.png")
viz.scoreboard(info, save="figures/check_score.png")
viz.learning_curve(info, save="figures/check_curve.png")
plt.show()

### Done

`weights/galaxy_sr.pt` is all the demo notebook needs. Open `02_demo.ipynb`
and run it once end-to-end today, so that tomorrow you are re-running
something you have already seen work.